# Klasik ML Algoritma Seçimi

Bu notebook, BBC ses arşivinin 467-boyutlu özellik vektörü üzerinde **8 farklı klasik makine öğrenmesi algoritmasını** karşılaştırarak hangi algoritmanın bu probleme en uygun olduğunu gösterir.

**Amaç:** LightGBM seçimini sezgiye değil, veriye dayandırmak.

| # | Algoritma | Tip | Beklenti |
|---|-----------|-----|----------|
| 1 | Logistic Regression | Lineer | Taban çizgisi |
| 2 | k-Nearest Neighbors | Örnek tabanlı | Yüksek boyutta zayıf |
| 3 | Support Vector Machine | Kernel | İyi ama yavaş |
| 4 | Random Forest | Ensemble (bagging) | Güçlü |
| 5 | Extra Trees | Ensemble (rastgele) | RF'ten hızlı |
| 6 | Gradient Boosting | Ensemble (boosting) | İyi ama yavaş |
| 7 | XGBoost | Boosting | LightGBM rakibi |
| 8 | **LightGBM** | Boosting | **Hipotez: kazanan** |

> **Not:** kNN ve SVM büyük veri setlerinde çok yavaş olduğundan alt örneklem (15.000 segment) üzerinde eğitilir.

## 1. Ortam ve Bağımlılıklar

In [ ]:
import sys
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

ROOT = Path('f:/Drthesiscode')
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.ensemble import (ExtraTreesClassifier, RandomForestClassifier,
                               HistGradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                              confusion_matrix)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb
import xgboost as xgb

from src.data.labels import SINIFLAR

SEED = 42
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

SINIF_ADLARI = [SINIFLAR[s][0] for s in sorted(SINIFLAR.keys())]
print(f'Sinif sayisi : {len(SINIF_ADLARI)}')
print(f'lgb={lgb.__version__}  xgb={xgb.__version__}')

## 2. Veri Yükleme ve Bölme

**Dosya düzeyinde stratified split** uygulanır — aynı dosyanın segmentleri aynı kümede kalır, veri sızıntısı önlenir.

Oran: Train %70 / Val %15 / Test %15

In [ ]:
df = pd.read_parquet(ROOT / 'data/processed/lgbm_ozellikler.parquet')

ozellik_sutunlari = sorted(
    [c for c in df.columns if c.startswith('f') and c[1:].isdigit()],
    key=lambda c: int(c[1:])
)
print(f'Toplam segment : {len(df):,}')
print(f'Ozellik boyutu : {len(ozellik_sutunlari)}')
print(f'Sinif dagilimi :\n{df["sinif"].value_counts().head(10).to_string()}')

In [ ]:
# Dosya duzeyinde stratified split
dosya_sinif = df.groupby('dosya')['sinif_idx'].first().reset_index()

d_trainval, d_test = train_test_split(
    dosya_sinif, test_size=0.15,
    stratify=dosya_sinif['sinif_idx'], random_state=SEED)
d_train, d_val = train_test_split(
    d_trainval, test_size=0.15/0.85,
    stratify=d_trainval['sinif_idx'], random_state=SEED)

def filtrele(d): return df[df['dosya'].isin(d['dosya'])]

df_train = filtrele(d_train)
df_val   = filtrele(d_val)
df_test  = filtrele(d_test)

X_train = df_train[ozellik_sutunlari].values.astype(np.float32)
y_train = df_train['sinif_idx'].values
X_val   = df_val[ozellik_sutunlari].values.astype(np.float32)
y_val   = df_val['sinif_idx'].values
X_test  = df_test[ozellik_sutunlari].values.astype(np.float32)
y_test  = df_test['sinif_idx'].values

print(f'Train: {len(X_train):,} segment ({d_train["dosya"].nunique()} dosya)')
print(f'Val  : {len(X_val):,} segment ({d_val["dosya"].nunique()} dosya)')
print(f'Test : {len(X_test):,} segment ({d_test["dosya"].nunique()} dosya)')

## 3. Ön İşleme: Ölçeklendirme ve Sınıf Ağırlıkları

- **StandardScaler**: Ortalama 0, standart sapma 1 — SVM ve Logistic Regression için zorunlu, tree-based modeller için nötr
- **Sınıf ağırlıkları**: Dengesiz veri setinde azınlık sınıflarını penalize etmemek için `class_weight='balanced'`

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

siniflar   = np.unique(y_train)
agirliklar = compute_class_weight('balanced', classes=siniflar, y=y_train)
ag_map     = dict(zip(siniflar, agirliklar))
sw_train   = np.array([ag_map[y] for y in y_train])

print(f'Ortalama sinif agirlik: {np.mean(list(ag_map.values())):.3f}')
print(f'Min agirlik: {min(ag_map.values()):.3f}  Max: {max(ag_map.values()):.3f}')

# Sinif dagilimi gorseli
fig, ax = plt.subplots(figsize=(14, 4))
counts = df_train['sinif'].value_counts()
ax.bar(range(len(counts)), counts.values, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(counts.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Segment sayisi')
ax.set_title('Train kumesi sinif dagilimi', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Algoritma Tanımları

Her algoritma **varsayılan** veya literatürde ses sınıflandırması için önerilen parametrelerle çalıştırılır — HPO yapılmaz. Amaç adil bir karşılaştırma zemini kurmak.

In [ ]:
YAVAS_MAX = 15_000  # kNN ve SVM icin alt orneklem buyuklugu

# True: LR ve RF atla (onceki calisma: LR=0.3593, RF=0.4793) — sadece agac modelleri
HIZLI_MOD = True

algoritmalar_tam = [
    {
        'ad': 'Logistic Regression',
        'aciklama': 'Lineer siniflandirici, taban cizgisi',
        'model': LogisticRegression(
            max_iter=1000, C=1.0, class_weight='balanced',
            solver='lbfgs', multi_class='multinomial',
            random_state=SEED, n_jobs=-1),
        'alt_orneklem': False,
    },
    {
        'ad': 'k-NN (k=5)',
        'aciklama': f'Ornek tabanli, alt orneklem ({YAVAS_MAX:,})',
        'model': KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1),
        'alt_orneklem': True,
    },
    {
        'ad': 'SVM (RBF)',
        'aciklama': f'Kernel SVM, alt orneklem ({YAVAS_MAX:,})',
        'model': SVC(
            C=10.0, kernel='rbf', gamma='scale',
            class_weight='balanced', decision_function_shape='ovr',
            random_state=SEED),
        'alt_orneklem': True,
    },
    {
        'ad': 'Random Forest',
        'aciklama': '300 agac, bagging ensemble',
        'model': RandomForestClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=SEED, n_jobs=-1),
        'alt_orneklem': False,
    },
    {
        'ad': 'Extra Trees',
        'aciklama': '300 agac, rastgele bolme noktalari',
        'model': ExtraTreesClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=SEED, n_jobs=-1),
        'alt_orneklem': False,
    },
    {
        'ad': 'Hist Gradient Boosting',
        'aciklama': 'Histogram tabanli GB, cok daha hizli (sklearn)',
        'model': HistGradientBoostingClassifier(
            max_iter=300, max_depth=8, learning_rate=0.1,
            random_state=SEED),
        'alt_orneklem': False,
    },
    {
        'ad': 'XGBoost',
        'aciklama': '300 agac, GPU (CUDA)',
        'model': xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            device='cuda', tree_method='hist',
            eval_metric='mlogloss', random_state=SEED,
            verbosity=0),
        'alt_orneklem': False,
    },
    {
        'ad': 'LightGBM',
        'aciklama': '500 agac, GPU',
        'model': lgb.LGBMClassifier(
            n_estimators=500, num_leaves=127, max_depth=8,
            learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
            class_weight='balanced', device='gpu',
            random_state=SEED, n_jobs=-1, verbose=-1),
        'alt_orneklem': False,
    },
]

HIZLI_ATLA = {'Logistic Regression', 'Random Forest', 'k-NN (k=5)', 'SVM (RBF)'}
algoritmalar = [a for a in algoritmalar_tam if not HIZLI_MOD or a['ad'] not in HIZLI_ATLA]

if HIZLI_MOD:
    print('HIZLI_MOD=True: LR/RF/kNN/SVM atlandi (bilinen sonuclar: LR=0.3593, RF=0.4793)')

print(f'{len(algoritmalar)} algoritma calistirilacak:')
for a in algoritmalar:
    etiket = ' *alt orneklem' if a['alt_orneklem'] else ''
    print(f"  {a['ad']:<28} — {a['aciklama']}{etiket}")

## 5. Eğitim ve Değerlendirme

Her algoritma için:
1. Eğit (gerekirse alt örneklem)
2. Val ve Test kümelerinde Macro F1 + Accuracy hesapla
3. Eğitim süresini kaydet

In [ ]:
sonuclar = []

for alg in algoritmalar:
    ad    = alg['ad']
    model = alg['model']

    if alg['alt_orneklem'] and len(X_train_s) > YAVAS_MAX:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(X_train_s), YAVAS_MAX, replace=False)
        X_tr, y_tr = X_train_s[idx], y_train[idx]
        sw_tr = None
    else:
        X_tr, y_tr, sw_tr = X_train_s, y_train, sw_train

    print(f'[{ad}] egitiliyor...', end=' ', flush=True)
    t0 = time.time()

    try:
        if sw_tr is not None:
            try:
                model.fit(X_tr, y_tr, sample_weight=sw_tr)
            except TypeError:
                model.fit(X_tr, y_tr)
        else:
            model.fit(X_tr, y_tr)
    except Exception as e:
        print(f'HATA: {e}')
        continue

    sure = time.time() - t0

    val_pred  = model.predict(X_val_s)
    test_pred = model.predict(X_test_s)

    val_f1   = f1_score(y_val,  val_pred,  average='macro', zero_division=0)
    test_f1  = f1_score(y_test, test_pred, average='macro', zero_division=0)
    val_acc  = accuracy_score(y_val,  val_pred)
    test_acc = accuracy_score(y_test, test_pred)

    print(f'Val F1={val_f1:.4f}  Test F1={test_f1:.4f}  Sure={sure:.1f}s')

    sonuclar.append({
        'Algoritma':       ad,
        'Val Macro F1':    round(val_f1,  4),
        'Test Macro F1':   round(test_f1, 4),
        'Val Accuracy':    round(val_acc, 4),
        'Test Accuracy':   round(test_acc,4),
        'Sure (sn)':       round(sure, 1),
        'Alt Orneklem':    alg['alt_orneklem'],
        '_model':          model,
        '_test_pred':      test_pred,
    })

print('\nTum algoritmalar tamamlandi.')

## 6. Sonuç Tablosu

In [ ]:
df_sonuc = pd.DataFrame([
    {k: v for k, v in s.items() if not k.startswith('_')}
    for s in sonuclar
]).sort_values('Val Macro F1', ascending=False).reset_index(drop=True)

df_sonuc.index += 1  # 1'den baslat

# Kazanani vurgula
kazanan_ad = df_sonuc.iloc[0]['Algoritma']

print(f'\n{"="*65}')
print('  ALGORITMA KARSILASTIRMASI — Val Macro F1 siralamasi')
print(f'{"="*65}')
print(df_sonuc[['Algoritma','Val Macro F1','Test Macro F1',
                'Val Accuracy','Sure (sn)']].to_string())
print(f'\n  Kazanan: {kazanan_ad}')
print(f'  Val F1 : {df_sonuc.iloc[0]["Val Macro F1"]:.4f}')
print(f'  Test F1: {df_sonuc.iloc[0]["Test Macro F1"]:.4f}')

# Kaydet
df_sonuc.drop(columns=['Alt Orneklem']).to_csv(
    OUTPUTS / 'model_selection_results.csv', index=True)
print('\n  Tablo kaydedildi: outputs/model_selection_results.csv')

## 7. Görsel Karşılaştırma

In [ ]:
df_plt = df_sonuc.sort_values('Val Macro F1', ascending=True)
renkler = ['#e74c3c' if a == kazanan_ad else '#3498db'
           for a in df_plt['Algoritma']]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Sol: Val F1
bars = axes[0].barh(df_plt['Algoritma'], df_plt['Val Macro F1'],
                     color=renkler, edgecolor='white', height=0.6)
for bar, val in zip(bars, df_plt['Val Macro F1']):
    axes[0].text(bar.get_width() + 0.004, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].axvline(df_plt['Val Macro F1'].max(), color='red',
                linestyle='--', alpha=0.4, linewidth=1.5)
axes[0].set_xlim(0, df_plt['Val Macro F1'].max() * 1.18)
axes[0].set_xlabel('Macro F1', fontsize=11)
axes[0].set_title('Validasyon Macro F1', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)

# Sag: Egitim suresi
sure_renk = ['#e67e22' if a['Alt Orneklem'] else '#95a5a6'
             for _, a in df_plt.iterrows()]
bars2 = axes[1].barh(df_plt['Algoritma'], df_plt['Sure (sn)'],
                      color=sure_renk, edgecolor='white', height=0.6)
for bar, (_, satir) in zip(bars2, df_plt.iterrows()):
    etiket = f"{satir['Sure (sn)']:.1f}s"
    if satir['Alt Orneklem']:
        etiket += ' *'
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 etiket, va='center', fontsize=9)
axes[1].set_xlabel('Saniye', fontsize=11)
axes[1].set_title('Egitim Suresi', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle(
    f'Klasik ML Algoritma Secimi — BBC 26-Sinif (467-dim)\n'
    f'Kazanan: {kazanan_ad}  |  * Alt orneklem ({YAVAS_MAX:,} segment)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig(OUTPUTS / 'model_selection_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik kaydedildi: outputs/model_selection_chart.png')

## 8. Val F1 vs Eğitim Süresi — Verimlilik Analizi

İdeal model sağ üst köşede: yüksek F1, kısa süre.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for _, satir in df_sonuc.iterrows():
    renk = '#e74c3c' if satir['Algoritma'] == kazanan_ad else '#3498db'
    isaretci = 's' if satir['Alt Orneklem'] else 'o'
    ax.scatter(satir['Sure (sn)'], satir['Val Macro F1'],
               s=120, color=renk, marker=isaretci, zorder=3)
    ax.annotate(satir['Algoritma'],
                (satir['Sure (sn)'], satir['Val Macro F1']),
                textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_xlabel('Egitim Suresi (sn) — log olcek', fontsize=11)
ax.set_ylabel('Val Macro F1', fontsize=11)
ax.set_title('F1 - Sure Dengesi\n(kare = alt orneklem kullanildi)', fontsize=12)
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS / 'model_selection_scatter.png', dpi=150)
plt.show()

## 9. Kazanan Algoritmanın Sınıf Bazlı Analizi

In [ ]:
# Kazananin test tahminlerini al
kazanan_sonuc = next(s for s in sonuclar if s['Algoritma'] == kazanan_ad)
test_pred_kazanan = kazanan_sonuc['_test_pred']

# Sinif bazli F1 raporu
rapor = classification_report(
    y_test, test_pred_kazanan,
    target_names=SINIF_ADLARI,
    zero_division=0, output_dict=True
)
print(f'Kazanan: {kazanan_ad}')
print(classification_report(
    y_test, test_pred_kazanan,
    target_names=SINIF_ADLARI,
    zero_division=0, digits=3
))

In [ ]:
# Sinif bazli F1 bar grafigi
sinif_f1 = {ad: rapor[ad]['f1-score']
            for ad in SINIF_ADLARI if ad in rapor}
siralanmis = dict(sorted(sinif_f1.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(12, 7))
renkler_bar = ['#e74c3c' if v < 0.5 else '#f39c12' if v < 0.7 else '#2ecc71'
               for v in siralanmis.values()]
ax.barh(list(siralanmis.keys()), list(siralanmis.values()),
        color=renkler_bar, edgecolor='white')
ax.axvline(np.mean(list(sinif_f1.values())), color='navy',
           linestyle='--', linewidth=1.5, label=f'Ortalama F1')
ax.set_xlabel('F1 Skoru')
ax.set_title(f'{kazanan_ad} — Sinif Bazli F1 (Test Kumesi)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(OUTPUTS / 'kazanan_sinif_f1.png', dpi=150)
plt.show()

zayif = [ad for ad, f1 in sinif_f1.items() if f1 < 0.5]
print(f'\nF1 < 0.5 olan siniflar ({len(zayif)} adet) — uretken dongu hedefleri:')
for ad in sorted(zayif, key=lambda x: sinif_f1[x]):
    print(f'  {ad:<35} F1={sinif_f1[ad]:.3f}')

## 10. Sonuçların Kaydedilmesi ve Yorum

Aşağıdaki hücre tüm sonuçları JSON olarak kaydeder ve bir sonraki adım için öneride bulunur.

In [ ]:
rapor_json = {
    'kazanan': kazanan_ad,
    'kazanan_val_f1':  df_sonuc.iloc[0]['Val Macro F1'],
    'kazanan_test_f1': df_sonuc.iloc[0]['Test Macro F1'],
    'zayif_siniflar': [
        {'sinif': ad, 'f1': round(sinif_f1[ad], 4)}
        for ad in sorted(zayif, key=lambda x: sinif_f1[x])
    ],
    'tum_sonuclar': df_sonuc[[
        'Algoritma', 'Val Macro F1', 'Test Macro F1',
        'Val Accuracy', 'Sure (sn)'
    ]].to_dict(orient='records'),
}

with open(OUTPUTS / 'model_selection_report.json', 'w', encoding='utf-8') as f:
    json.dump(rapor_json, f, ensure_ascii=False, indent=2)

print('Rapor kaydedildi: outputs/model_selection_report.json')
print()
print('=' * 60)
print(f'  OZET')
print('=' * 60)
print(f'  Kazanan algoritma : {kazanan_ad}')
print(f'  Val Macro F1      : {df_sonuc.iloc[0]["Val Macro F1"]:.4f}')
print(f'  Test Macro F1     : {df_sonuc.iloc[0]["Test Macro F1"]:.4f}')
print(f'  Zayif sinif sayisi: {len(zayif)} (F1 < 0.5)')
print()
print('  Sonraki adimlar:')
print(f'  1. {kazanan_ad} icin Optuna HPO:')
print(f'     python scripts/02_train_lgbm.py --trials 50')
print(f'  2. Zayif siniflar icin uretken veri artirma (Asama 3)')
print('=' * 60)